Welcome to the deployment notebook in which we will run a temporal server that will allow us to test the model for inference.

The reason why I used a Kaggle notebook to build the REST API for the model is because the model requires GPUs to run, and Kaggle offers 2, so I managed to find the way to take advantage of those GPUs to be able to test the model "locally" with a basic React interface I built.

This model is finetuned to only give diet recommendations, it can not chat with you or follow up a conversation, it only provides an output for each prompt that we provide to the model, which itself takes time to reason.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

We install all the dependencies necessary to run everything.

Look at how many Backend Python libraries are installed, because as I said, we will "run" a temporal server.

In [ ]:
!pip install bitsandbytes torch transformers fastapi uvicorn pyngrok asyncio nest_asyncio

We use quantization again, to make sure the model loads and fits into the limited RAM we have.

In [ ]:
from transformers import BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

We end up loading the model, but also offloading it to the disk, because the base model with the LoRA adapter are already too big to even fit in the RAM, which forces us to load the model separately, offloading it into disk, then the tokenizer, and finally we create the pipeline to test the model for inference.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

model = AutoModelForCausalLM.from_pretrained(
    "sse3ele3/VictuAI-v5",
    torch_dtype=torch.float16,
    device_map="auto",
    offload_folder="/kaggle/working/offload"
)

tokenizer = AutoTokenizer.from_pretrained("sse3ele3/VictuAI-v5")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

We build the functions necessary for the model to operate, and also we have a system prompt which guides the model to provide a concrete response to what is being asked.

The format_prompt function is to convert the prompt provided by the user into a proper instruction in Qwen chat template format to make sure the model can perfectly encode the input and provide a meaningful answer, in this case a personalised diet recommendation.

The predict function is the most important function, it produces the output that we will provide to the user. First we call the format_prompt function and then we produce the output by guiding the model into providing a response of 1500 tokens, which lets it think as much as he wants, because Deepseek R1 models reason before giving a response, which we expect to be deterministic, long and complete, choosing most of the time the most probable token, and never stopping early.

The generate_chunks function basically allows us to stream the response, meaning that we return word for word instead of the plain text all at once, which is more pleasing for the UX.

In [ ]:
import asyncio

SYSTEM_PROMPT = """Act as a nutrition expert. Give ONLY the final meal plan. Do not explain your reasoning.
                Do not think step by step. Start immediately with "Daily Meal Plan:"""

def format_prompt(prompt):
    input = f"""<|im_start|>system\n{SYSTEM_PROMPT}\n<|im_end|>\n\n
                <|im_start|user\n{prompt}\n<|im_end|>\n\n
                <|im_start|>assistant"""
    return input

def predict(prompt):
    input = format_prompt(prompt)
    output = pipe(  input,
                    max_new_tokens=1500,           
                    temperature=0.1,               
                    do_sample=True,
                    top_p=0.95,
                    top_k=50,
                    repetition_penalty=1.05,       
                    length_penalty=1.2,            
                    early_stopping=False,          
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    return_full_text=False,
                    num_return_sequences=1,
                    remove_invalid_values=True
                )[0]["generated_text"]
    return output.split("</think>")[-1].strip()

async def generate_chunks(output: str, delay: float = 0.05):
    for word in output.split(" "):
        yield f"""data: {json.dumps({"token": word})}\n\n"""
        await asyncio.sleep(delay)

Using the previous functions, we build the REST API in FastAPI, with 3 endpoints, chat, chat/stream and health.

First of all, we establish a CORS middleware, which is very permissive and not safe on production, and lets us "run" the server in one URL, like localhost:8000, and then have our frontend page on another URL, like localhost:3000, and the browser doesn't block it.

The chat endpoint allows us the test the model for inference by sending prompts and getting responses.

The chat/stream endpoint is the same as the chat endpoint, but it make streams the word as I mentioned before.

The health endpoint just says if the model is running, which only happens when the "server" is running.

In [ ]:
# REST API
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

@app.post("/chat")
def chat(prompt: str):
    output = predict(prompt) 
    return {"response": output}

@app.post("/chat/stream")
def chat_stream(prompt: str):
    output = predict(prompt)
    return StreamingResponse(
        generate_chunks(output),
        media_type="text/event-stream"
    )
    
@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "model": "VictuAI-v5",
        "ready": true
    }

Here we log in to ngrok.

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("xxxxxxxxxxxxxxxxxxxxx")

Finally, we start the "server".

The nest_asyncio library allows us to run nested loops, which we need because we are in a notebook and we want to server to run for a period of time, not for just one execution.

Then we get a public URL from ngrok which we put into the App.js in our React app, and for last we run with uvicorn the "server" for an indefinite period of time, even though the maximum time allowed is 9 hours in free tier Kaggle notebooks.

In [ ]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn

nest_asyncio.apply()

public_url = ngrok.connect(8000)
print(f"URL: {public_url}")

uvicorn.run(app, host="0.0.0.0", port=8000)